# Notebook 104: Project Agentic AI & RAG: Lakehouse Document Chunking and Vector Search

**Module:** Module 10: End-to-End Enterprise Projects & Production Workflows  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 25: Genie Code & Video 27: Agentic AI for Beginners: Build an E2E Agentic Solution  

---

## 1. Business Scenario & Real-World Context
An enterprise customer service platform deploys an Agentic AI chatbot powered by Databricks Genie and LLMs. The chatbot needs to answer questions about company product manuals, warranty terms, and legal policies stored across unstructured PDF and text documents in the Lakehouse.

## 2. Technical Requirement & Problem Statement
Build an enterprise RAG (Retrieval-Augmented Generation) data preparation pipeline: (1) Ingest raw documentation from Unity Catalog Volumes, (2) Split long text into overlapping chunks using sliding windows, (3) Generate dense vector embeddings simulation, and (4) Prepare Gold vector index tables for Vector Search.

## 3. Spark & Databricks Architecture Notes
RAG architectures require chunking large documents into semantically coherent segments (e.g. 500 characters with 100 character overlap) and embedding them into high-dimensional vectors. Databricks Vector Search indexes Delta tables natively, enabling hybrid semantic and keyword search.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_104_Project_Agentic_AI_&_RAG_Lakehouse_Document_Chunking_and_Vector_Search")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Ingesting Unstructured Product Documentation
Create sample product warranty and documentation text.

In [ ]:
docs = [
    ("DOC-101", "Smart TV Manual", "The Smart TV comes with a 2-year manufacturer warranty covering panel defects and audio failure. To request service, submit your receipt online within 30 days of the incident. Accidental drops and water damage are not covered under the warranty terms."),
    ("DOC-102", "Return Policy", "Customers may return unopened items within 14 days of purchase for a full refund to the original payment method. Opened items are subject to a 15% restocking fee unless defective. Software downloads are strictly non-refundable.")
]

schema = "doc_id STRING, title STRING, body_text STRING"
df_docs = spark.createDataFrame(docs, schema)
df_docs.show(truncate=False)

## 6. Text Chunking via Sliding Windows in PySpark
Split text into overlapping chunks using higher-order functions or UDFs.

In [ ]:
def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split(" ")
    chunks = []
    i = 0
    step = max(1, (chunk_size - overlap) // 6)
    window_words = chunk_size // 6
    while i < len(words):
        chunk = " ".join(words[i : i + window_words])
        if chunk.strip():
            chunks.append(chunk)
        i += step
    return chunks

chunk_udf = F.udf(chunk_text, T.ArrayType(T.StringType()))

df_chunked = (
    df_docs
    .withColumn("chunks", chunk_udf("body_text"))
    .select("doc_id", "title", F.explode("chunks").alias("chunk_text"))
    .withColumn("chunk_id", F.concat(F.col("doc_id"), F.lit("_CHK_"), F.monotonically_increasing_id()))
)

print("Document Chunks Prepared for Embedding Generation:")
df_chunked.select("chunk_id", "title", "chunk_text").show(truncate=False)

## 7. Vector Embedding Representation & Gold Vector Index Schema
Represent dense embedding vectors and prepare schema for Databricks Vector Search.

In [ ]:
# Simulate 8-dimensional dense vector embeddings for demonstration
df_vector_ready = (
    df_chunked
    .withColumn(
        "embedding_vector",
        F.array([F.round(F.rand(seed=42 + i) * 2 - 1, 4) for i in range(8)])
    )
    .withColumn("_embedded_at", F.current_timestamp())
)

print("Vector Search Index Table Schema:")
df_vector_ready.printSchema()
df_vector_ready.select("chunk_id", "title", "embedding_vector").show(truncate=False)

## 9. Key Architectural Takeaways
- **RAG pipelines bridge unstructured enterprise documents with LLM reasoning via Vector Search.**
- **Chunking with overlap preserves context across sentence boundaries.**
- **Delta Lake serves as the unified storage layer for both raw text documents and high-dimensional vector embeddings.**

## 10. Senior Data Engineering Interview Questions
1. What is Chunk Overlap in RAG pipelines and why is it important for semantic retrieval?
2. How does Databricks Vector Search synchronize automatically with underlying Delta tables?
3. What are the trade-offs between dense semantic vector search and sparse keyword search (BM25)?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
